# Microsoft Fabric × DuckDB × openaivec — Structured PDF Extraction (Inventory Transfer Slips)

This notebook is designed to run inside a **Microsoft Fabric Notebook** and builds the following pipeline:

1. Read inventory transfer slip PDFs from `Files/transfer/*.pdf` of a Fabric Lakehouse via [DuckDB](https://duckdb.org/).
2. Use [openaivec](https://github.com/microsoft/openaivec)'s `responses_udf` to parse each PDF into a Pydantic schema with an Azure OpenAI multimodal model.
3. Convert the result to a pandas DataFrame and write it to Delta Lake on OneLake.

`responses_udf` takes care of batching, deduplication, retry, and structured output (Pydantic `response_format`), so your code is essentially **just SQL + a Pydantic schema**.

---

## Prerequisites

### 1. Fabric / OneLake

- This notebook is placed in a **Fabric workspace**.
- A **Lakehouse** (e.g. `bronze.Lakehouse`) is created and attached as the default Lakehouse of this notebook.
- Input PDFs are uploaded to `Files/transfer/` of that Lakehouse (accessible from the kernel via the POSIX path `/lakehouse/default/Files/transfer/*.pdf`).
- The Delta write target (e.g. `abfss://<workspace>@onelake.dfs.fabric.microsoft.com/<lakehouse>.Lakehouse/Tables/...`) is writable.

### 2. Azure OpenAI / AI Foundry

- An Azure OpenAI or AI Foundry endpoint in **v1 API format**:
  - Example: `https://<your-resource>.services.ai.azure.com/openai/v1/`
  - Foundry project variant: `https://<your-resource>.services.ai.azure.com/api/projects/<project>/openai/v1/`
- A **multimodal-capable deployment** (e.g. a `gpt-4.1-mini` family model) is deployed under that endpoint.

### 3. Entra ID authentication (Service Principal + Key Vault, accessed via the Fabric Workspace identity)

Fabric Notebooks do not support `DefaultAzureCredential` directly, so the Service Principal's `clientSecret` is retrieved from Key Vault and a `ClientSecretCredential` is constructed. openaivec's `_provider.py` performs this automatically when `KEY_VAULT_URL` and `KEY_VAULT_SECRET_NAME` are set.

**Important — Key Vault access uses the Fabric Workspace identity**: under the hood, `notebookutils.credentials.getSecret(...)` calls Key Vault as the **Fabric workspace** (identified by its Workspace ID), **not** as the Service Principal you are about to use against Azure OpenAI, and **not** as the interactive user. You therefore need to grant Key Vault read permission to the workspace itself.

| Item | Details |
| --- | --- |
| Service Principal (SP) | Create an App Registration; note its `tenantId` and `clientId`. |
| SP role on Azure OpenAI / Foundry | Grant the SP the **`Cognitive Services OpenAI User`** role on the AI resource. |
| Key Vault — store secret | Store the SP's `clientSecret` as a Key Vault secret. |
| Key Vault — RBAC (workspace) | Grant **`Key Vault Secrets User`** to the **Fabric workspace identity** (object id = Fabric Workspace ID). This is the principal that `notebookutils.credentials.getSecret` impersonates. |
| Key Vault access model | Use **Azure RBAC** (not legacy access policies) on the Key Vault so the workspace assignment is honored. |

> How to find the Fabric Workspace ID: open the workspace in the Fabric portal, click *Workspace settings → About*, and copy the **Workspace ID** (GUID). Assign roles against this object id in Key Vault's *Access control (IAM)*.

> **Security**: all Tenant ID / Client ID / Key Vault URI / endpoint values in the cells below are **placeholders**. Replace them with your own values and never commit real secrets to a repository.


## 1. Install dependencies

Uninstall the older `openai` shipped with the Fabric runtime, then install `openaivec` (which pulls in the latest `openai`) and `azure-identity`.


In [ ]:
%pip uninstall -y openai
%pip install -U openaivec
%pip install -U azure-identity


Found existing installation: openai 1.109.1
Uninstalling openai-1.109.1:
  Successfully uninstalled openai-1.109.1
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 38.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 112.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 30.4 MB/s  0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.3.3
    Uninstalling pandas-2.3.3:0m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/4 [pandas]
   ━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/4 [pandas]

## 2. Configure environment variables (placeholders)

openaivec evaluates the following environment variables to decide how to authenticate:

| Variable | Purpose |
| --- | --- |
| `AZURE_TENANT_ID` | Tenant ID of the Service Principal used against Azure OpenAI. |
| `AZURE_CLIENT_ID` | Client ID of that Service Principal. |
| `KEY_VAULT_URL` | URL of the Key Vault that stores the SP's client secret. |
| `KEY_VAULT_SECRET_NAME` | Name of the secret inside that Key Vault. |
| `AZURE_OPENAI_BASE_URL` | Azure OpenAI / Foundry base URL in **v1 format** (must end with `/openai/v1/`). |

Inside Fabric, openaivec detects `notebookutils`, calls `notebookutils.credentials.getSecret(KEY_VAULT_URL, KEY_VAULT_SECRET_NAME)` **as the Fabric workspace identity**, and then builds a `ClientSecretCredential` (or its async equivalent from `azure.identity.aio`) from the resulting secret. Make sure the Fabric workspace itself has been granted `Key Vault Secrets User` on the vault — see the prerequisites above.

> Replace every `<...>` placeholder with your own value before running the next cell.


In [ ]:
import os

# --- Service Principal used against Azure OpenAI / Foundry ---
os.environ["AZURE_TENANT_ID"] = "<YOUR_TENANT_ID>"
os.environ["AZURE_CLIENT_ID"] = "<YOUR_SP_CLIENT_ID>"

# --- Key Vault that stores the SP client secret.
#     The Fabric workspace identity (Workspace ID) must have
#     `Key Vault Secrets User` on this vault. ---
os.environ["KEY_VAULT_URL"] = "https://<your-keyvault>.vault.azure.net/"
os.environ["KEY_VAULT_SECRET_NAME"] = "<your-secret-name>"

# --- Azure OpenAI / Foundry endpoint (v1 format) ---
os.environ["AZURE_OPENAI_BASE_URL"] = (
    "https://<your-resource>.services.ai.azure.com/openai/v1/"
)


## 3. Import libraries and initialize openaivec

All symbols used by the notebook are imported here in one place. Importing `pandas_ext` registers the `.ai` / `.aio` accessors on pandas `Series` and `DataFrame`. The final line runs a tiny smoke test (English → French translation) to confirm that authentication, model selection, and network reachability all work end-to-end.

> Replace the model name with one of your **multimodal-capable** deployments — it is required because the pipeline feeds PDF bytes directly to the model.


In [ ]:
# Standard library
from enum import Enum

# Third-party
import duckdb
import pandas as pd
from deltalake.writer import write_deltalake
from pydantic import BaseModel

# openaivec
import openaivec
from openaivec import pandas_ext  # noqa: F401  registers the .ai / .aio accessor
from openaivec.duckdb_ext import responses_udf

# Name of a deployed multimodal-capable model
openaivec.set_responses_model("<YOUR_MULTIMODAL_DEPLOYMENT>")

# Smoke test: if this returns a French translation,
# auth + model + network are all working.
pd.Series(["apple", "banana"]).ai.responses("Translate to French.")


Processing batches:   0%|          | 0/2 [00:00<?, ?item/s]

0     pomme
1    banane
dtype: str

## 4. Open an in-memory DuckDB connection

DuckDB is a single-process OLAP engine. It runs entirely inside the Fabric Notebook's Python kernel and can read OneLake files directly via the POSIX path that Fabric mounts under `/lakehouse/default/Files/...`.

We create one connection that will host the UDF registration and all subsequent SQL.


In [ ]:
conn = duckdb.connect()


## 5. Define the extraction schema (Pydantic)

We declare *what to extract from each PDF* as a **Pydantic model**. `responses_udf` forwards this schema to Azure OpenAI as a Structured Output (`response_format`), which gives us:

- the LLM can only return JSON that conforms to the schema,
- values are automatically validated and converted to Python types,
- `Enum` fields constrain allowed values to a fixed set.

For this sample we model an inventory transfer slip as `InventoryTransferSlip`, along with its line items (`TransferItem`) and two enums.


In [ ]:
class InventoryCategory(str, Enum):
    """Inventory item category."""
    RAW_MATERIAL = "原材料"
    PARTS = "部品"
    FINISHED_GOODS = "製品"
    PACKAGING = "梱包資材"
    CONSUMABLES = "消耗品"


class TransferStatus(str, Enum):
    """Slip status."""
    PENDING = "申請中"
    APPROVED = "承認済"
    IN_TRANSIT = "移動中"
    COMPLETED = "完了"
    PARTIAL = "一部完了"


class TransferItem(BaseModel):
    """A single line item on a transfer slip."""
    sku: str
    name: str
    category: InventoryCategory
    unit: str
    quantity: int
    unit_price: float
    amount: float


class InventoryTransferSlip(BaseModel):
    """Structured representation of an inventory transfer slip."""
    slip_number: str
    date: str
    source_warehouse_code: str
    source_warehouse_name: str
    destination_warehouse_code: str
    destination_warehouse_name: str
    department: str
    requester: str
    approver: str
    reason: str
    status: TransferStatus
    items: list[TransferItem]
    total_quantity: int
    total_amount: float


## 6. Register `parse_transfer_slip` as a DuckDB UDF

`openaivec.duckdb_ext.responses_udf` registers a scalar function on a DuckDB connection that calls the Azure OpenAI Responses API. Key parameters:

- `instructions`: system prompt describing **what** to extract and **how** to format it.
- `response_format`: the Pydantic model from the previous cell — LLM output is guaranteed to match it.
- `batch_size=1`: one PDF per request (multimodal inputs are most reliable when sent individually).
- `max_concurrency=36`: number of parallel workers; tune against your Azure OpenAI rate limit.
- `multimodal=True`: required flag telling the UDF to treat the input column as files (PDF / images).

After registration you can simply call `SELECT parse_transfer_slip(file) FROM glob(...)` — it behaves like any other SQL function.


In [ ]:
responses_udf(
    conn,
    "parse_transfer_slip",
    instructions=(
        "You are a parser for inventory transfer slips. "
        "Extract all information from the slip and return it in the structured format. "
        "Numbers must not contain currency symbols or thousand separators. "
        "Dates must be formatted as YYYY-MM-DD. "
        "Split warehouse codes (e.g. 'WH-TK01') and warehouse names into separate fields. "
        "Classify each line item into the appropriate inventory category."
    ),
    response_format=InventoryTransferSlip,
    batch_size=1,
    max_concurrency=36,
    multimodal=True,
)


## 7. List the input PDF files

The `glob()` table function enumerates PDFs under `Files/transfer/` of the attached Lakehouse. OneLake is mounted on the Fabric kernel's local filesystem, so the POSIX path `/lakehouse/default/Files/transfer/*.pdf` works directly.


In [8]:
conn.sql("""
    select
        *
    from glob('/lakehouse/default/Files/transfer/*.pdf')
""")

┌────────────────────────────────────────────────────┐
│                        file                        │
│                      varchar                       │
├────────────────────────────────────────────────────┤
│ /lakehouse/default/Files/transfer/transfer_001.pdf │
│ /lakehouse/default/Files/transfer/transfer_002.pdf │
│ /lakehouse/default/Files/transfer/transfer_003.pdf │
│ /lakehouse/default/Files/transfer/transfer_004.pdf │
│ /lakehouse/default/Files/transfer/transfer_005.pdf │
│ /lakehouse/default/Files/transfer/transfer_006.pdf │
│ /lakehouse/default/Files/transfer/transfer_007.pdf │
│ /lakehouse/default/Files/transfer/transfer_008.pdf │
│ /lakehouse/default/Files/transfer/transfer_009.pdf │
│ /lakehouse/default/Files/transfer/transfer_010.pdf │
│                         ·                          │
│                         ·                          │
│                         ·                          │
│ /lakehouse/default/Files/transfer/transfer_041.pdf │
│ /lakehou

## 8. Parse the PDFs and flatten line items

This is the core of the notebook. The SQL is structured as three CTEs:

1. **`parsed`**: call `parse_transfer_slip(file)` on each PDF, producing one `InventoryTransferSlip` STRUCT column.
2. **`items`**: `unnest()` the `items` list (`list[TransferItem]`) so each line item becomes its own row.
3. **Final SELECT**: join header columns with item columns on `slip_number` to build a flat one-row-per-item table.

The result is materialized as a pandas `DataFrame` via `.to_df()` and handed off to the Delta writer. `responses_udf` handles deduplication, auto-batching, and retries underneath — your code only needs to express the SQL.


In [19]:
pandas_dataframe = conn.sql("""
    with parsed as (
        select
            parse_transfer_slip(file) as r
        from glob('/lakehouse/default/Files/transfer/*.pdf')
    ), items as (
        select
            r.slip_number,
            unnest(r.items) as item
        from parsed
    )
    select
        r.slip_number,
        r.date,
        r.source_warehouse_code,
        r.source_warehouse_name,
        r.destination_warehouse_code,
        r.destination_warehouse_name,
        r.department,
        r.requester,
        r.approver,
        r.reason,
        r.status,
        items.item.sku as item_sku,
        items.item.name as item_name,
        items.item.category as item_category,
        items.item.unit as item_unit,
        items.item.quantity as item_quantity,
        items.item.unit_price as item_unit_price,
        items.item.amount as item_amount
    from parsed
    left join items
    on parsed.r.slip_number = items.slip_number

""").to_df()

In [10]:
pandas_dataframe

NameError: name 'pandas_dataframe' is not defined

## 9. Write to Delta Lake on OneLake

Save the extracted DataFrame to the attached Lakehouse under `Tables/transfers` as a Delta table. Substitute your own workspace and Lakehouse names into the `abfss://` URI. Once written, the table appears in the Lakehouse Explorer and is queryable from the SQL endpoint, Power BI, and other notebooks.


In [ ]:
write_deltalake(
    "abfss://<your-workspace>@onelake.dfs.fabric.microsoft.com/<your-lakehouse>.Lakehouse/Tables/transfers",
    pandas_dataframe,
    mode="overwrite",
)
